# Driver — benchmark sin-GT de las 6 configuraciones

Corre las **6 combinaciones** detector × tracker sobre los **5 videos seeded del
testing** y emite la **tabla comparativa** (eficiencia + trayectoria + máscara).

Corre **en el pod** (SAM3 + GPU). Procesa **una config a la vez** (corre → calcula
métricas con los JSON frescos → siguiente, con `overwrite=True`) para evitar que las
configs se pisen en `outputs/inference/<stem>/`.

Las métricas de trayectoria/máscara requieren `obj_id` estable ⇒ solo aplican a las 4
configs con tracking; las 2 de segmentación reportan solo eficiencia.

In [ ]:
from src.core.batch import run_batch
from src.eval.benchmark import (
    aggregate_config,
    benchmark_videos,
    comparison_table,
    write_table,
)

videos = benchmark_videos(n=5, seed=42)
print("videos del benchmark:")
for v in videos:
    print(" ", v)

# (label, mode, detector, tracker). tracker=None en las configs sin tracking.
CONFIGS = [
    ("sam3_text+none",     "segmentation", "sam3_text", None),
    ("sam3_text+bytetrack", "tracking",    "sam3_text", "bytetrack"),
    ("sam3_text+botsort",   "tracking",    "sam3_text", "botsort"),
    ("yolo_sam3+none",      "segmentation", "yolo_sam3", None),
    ("yolo_sam3+bytetrack", "tracking",    "yolo_sam3", "bytetrack"),
    ("yolo_sam3+botsort",   "tracking",    "yolo_sam3", "botsort"),
]

In [ ]:
# Una config a la vez: run_batch carga SAM3 internamente (una vez por config) y
# escribe los JSON; aggregate_config los lee ANTES de que la siguiente config los
# sobrescriba. include_masks=True para las métricas de máscara; overwrite=True para
# reprocesar en cada config.
rows = []
for label, mode, detector, tracker in CONFIGS:
    print(f"\n=== {label} ({mode}) ===")
    summary = run_batch(
        mode=mode,
        videos=videos,
        detector=detector,
        tracker=tracker,
        sampling="all",        # video completo (sin acotar frames)
        include_masks=True,
        render_video=False,
        overwrite=True,
    )
    row = aggregate_config(label, summary)
    rows.append(row)
    print(row)

In [ ]:
df = comparison_table(rows)
out = write_table(df)
print(f"tabla escrita en: {out}")
df

## Cómo leer la tabla

- **fps / peak_vram_mb** — eficiencia (más FPS y menos VRAM = mejor). El "filtro de la
  realidad".
- **tracklet_len** — mayor = menos fragmentación (IDs viven más).
- **frag_rate** — menor = menos cambios de ID sospechosos.
- **smoothness** — menor = trayectorias más físicas (menos saltos).
- **mask_iou** — mayor = máscaras más estables en el tiempo.
- **com_jitter** — menor = menos temblor del centro de masa.

Recordatorio: son métricas **sin ground-truth**. Miden consistencia/eficiencia, **no**
exactitud vs humano. La decisión final es humana.